In [1]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score

## Part 1. Loading the dataset

In [3]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
dataset = pd.read_csv("science_data_large.csv")
print(dataset.head(15))
dataset.info()

    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## Part 2. Splitting the dataset

In [4]:
# Take the pandas dataset and split it into our features (X) and label (y)
features = dataset.drop(columns=["Size nm^3"])
label = dataset["Size nm^3"]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size = 0.1, random_state=42)


## Part 3. Perform a Linear Regression

In [5]:
# Use sklearn to train a model on the training set

model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model

sampleInput = pd.DataFrame({'Temperature °C': [2.5], 'Mols KCL': [23.5]})
prediction = model.predict(sampleInput)
print(f"Preidcted output for the sample input {sampleInput.values}: {prediction[0]}")


# Report on the score for that model, in your own words (markdown, not code) explain what the score means

score = model.score(X_test, y_test)
print(f"Model R-squared score: {score: .2f}")

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX

coefficients = model.coef_
intercept = model.intercept_

equation = f"h(X) = {intercept: .2f} + {coefficients[0]: .2f} \\cdot KCI\\_amount + {coefficients[1]:.2f} \\cdot temperature"
print(equation)

Preidcted output for the sample input [[ 2.5 23.5]]: -382957.7794875616
Model R-squared score:  0.86
h(X) = -409391.48 +  866.15 \cdot KCI\_amount + 1032.70 \cdot temperature




$$
 y = -409391.48 +  866.15x_1 + 1032.70x_2
$$

With an R-squared score of 0.86, this means that about 86% of the variance in size of the slime can be explained by the temperature and the amount of KCl. 

## Part 4. Use Cross Validation

In [6]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
cv_scores = cross_val_score(model, features, label, cv=10)

# Report on their finding and their significance

print("Cross-Validation scores:", cv_scores)
print(f"Mean Cross-Validation Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation of Cross-Validation Scores: {cv_scores.std():.4f}")

Cross-Validation scores: [0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]
Mean Cross-Validation Score: 0.8552
Standard Deviation of Cross-Validation Scores: 0.0315


With a cross-validation score of [0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397 0.87941022 0.86349411 0.78353682 0.88686516], a mean cross-validation score of 0.8552, and a standard deviation score of 0.0315. With a low SD of 0.0315, it shows that the model is reliable since a low SD means that the model is consistent. Additionally, with a mean score above 0.85, it suggests that the model is performing well and is likely to generalize effectively to unseen data.

## Part 5. Using Polynomial Regression

In [7]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(features)

X_train_poly, X_test_poly, y_train, y_test = train_test_split(X_poly, label, test_size=0.1, random_state=42)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

poly_score = poly_model.score(X_test_poly, y_test)
print(f"Polynomial Model R-squared score: {poly_score:.2f}")

sampleInput_poly = pd.DataFrame({'Temperature °C': [2.5], 'Mols KCL': [23.5]})
sampleInput_poly_transformed = poly.transform(sampleInput_poly)

prediction_poly = poly_model.predict(sampleInput_poly_transformed)
print(f"Predicted output for the sample input {sampleInput_poly.values}: {prediction_poly[0]}")

coefficients_poly = poly_model.coef_
intercept_poly = poly_model.intercept_

equation_poly = "h(X) = {:.2f}".format(intercept_poly)
for i in range(1, len(coefficients_poly)):
    if i < 3:
        feature_name = "Temperature °C" if i == 1 else "Mols KCL"
        equation_poly += " + {:.2f} * {}".format(coefficients_poly[i], feature_name)
    else:
        feature_name = "emperature °C" if i == 3 else "Mols KCL"
        equation_poly += " + {:.2f} * {}".format(coefficients_poly[i], feature_name)

print("Polynomial regression equation:", equation_poly)

# Report on the metrics and output the resultant equation as you did in Part 3.

Polynomial Model R-squared score: 1.00
Predicted output for the sample input [[ 2.5 23.5]]: 163.27858893768197
Polynomial regression equation: h(X) = 0.00 + 12.00 * Temperature °C + -0.00 * Mols KCL + 0.00 * emperature °C + 2.00 * Mols KCL + 0.03 * Mols KCL


$$
    h(X) = 12x_1 + 2x_2 + 0.03x_3
$$

The polynomial model achieved a perfect R-squared score of 1.00, indicating it explains all variance in the target variable. However, this raises concerns about overfitting if evaluated on the same dataset used for training. For the input values of Temperature = 2.5 °C and Mols KCL = 23.5, the predicted output is approximately 163.28.